# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2:3b"


# if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
#     print("API key looks good so far")
# else:
#     print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
# MODEL = 'gpt-5-nano'
openai = OpenAI(base_url=OLLAMA_BASE_URL , api_key="ollama")

In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [69]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
Links will not be the subdomain domain will be same 
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [70]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [71]:
print(get_links_user_prompt("https://huggingface.co"))


Here is the list of links on the website https://huggingface.co -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

/
/models
/datasets
/spaces
/storage
/docs
/enterprise
/pricing
/tasks
/chat
/collections
/languages
/organizations
/blog
/posts
/papers
/hardware
/learn
/join/discord
https://discuss.huggingface.co/
https://github.com/huggingface
/enterprise
/pro
/support
/inference/models
/inference-endpoints
/storage
/login
/join
/spaces
/models
/deepseek-ai/DeepSeek-V4.1-Flash-1017
/RecursiveSI/flint
/XHToken/Spark-X2.6-4B
/Nanbeige/Nanbeige4.3-33B
/black-forest-labs/flux-3-image-klein-comfy
/models
/spaces/glitchlab/MiniMax-H3-Turbo-Lora-UNCENSORED-Workflow
/spaces/kulkas2pintu/wan777
/spaces/fishpasters/uncensored-imageTOimage2Ref
/spaces/hugging-apps/bfs-best-face-swap
/spaces/nvidia/simready-central
/sp

In [72]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [73]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about me page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'},
  {'type': 'careers/jobs page', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin page', 'url': 'https://www.linkedin.com/in/eddonner/'}]}

In [74]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [75]:
links = select_relevant_links("https://edwarddonner.com")

for link in links['links']:
    print(link['type'])


Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 3 relevant links
about page
 careers/jobs page
 twitter page


In [76]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 12 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/'},
  {'type': 'home page', 'url': 'https://huggingface.co/'},
  {'type': 'Models page', 'url': 'https://huggingface.co/models'},
  {'type': 'Datasets page', 'url': 'https://huggingface.co/datasets'},
  {'type': 'Spaces page', 'url': 'https://huggingface.co/spaces'},
  {'type': 'Pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'Blog page', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'Discord page', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Terms of Service', 'url': 'None'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [37]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [40]:
print(fetch_page_and_all_relevant_links("https://edwarddonner.com"))

Selecting relevant links for https://edwarddonner.com by calling llama3.2:3b
Found 6 relevant links
## Landing Page:

Home - Edward Donner

Skip to content
Avatar
Curriculum
Proficiency
C4
Outsmart
An arena that pits LLMs against each other in a battle of diplomacy and deviousness
About
Posts
Well, hi there.
I’m Ed. I like writing code and experimenting with LLMs, and hopefully you’re here because you do too. I also enjoy amateur electronic music production (
very
amateur) and losing myself in
Hacker News
, nodding my head sagely to things I only half understand.
I’m the co-founder and CTO of AI startup
Nebula.io
. I was previously founder and CEO of AI startup untapt,
acquired in 2021
, and a Managing Director at JPMorgan.
I will happily drone on for hours about LLMs to anyone in my vicinity. My friends got fed up with my impromptu lectures, and convinced me to make some Udemy courses. To my total joy (and shock) they’ve become best-selling, top-rated courses, with over a million enro

In [41]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [42]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [51]:
print(get_brochure_user_prompt("HuggingFace", "https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 11 relevant links

You are looking at a company called: HuggingFace
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.


## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
deepseek-ai/DeepSeek-V4.1-Flash-1017
Updated
9 days ago
•
95.9k
•
3.0

In [52]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [54]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 7 relevant links


Hugging Face: Building a Community of Innovation
==============================================

Welcome to Hugging Face, the premier platform for the machine learning community to collaborate on models, datasets, and applications. As a collaborative effort, we strive to provide the tools and resources necessary for innovation to thrive.

**Our Mission**
---------------

Hugging Face aims to make machine learning more accessible and user-friendly. We believe that by providing a platform for collaboration, innovation, and discovery, we can unlock the full potential of AI.

**Our Values**
--------------

*   **Community-driven**: We believe that the machine learning community is crucial to our success. We foster a collaborative environment where users can share knowledge, resources, and ideas.
*   **Innovation**: We strive to stay at the forefront of AI advancements and provide the tools and resources necessary for researchers, developers, and businesses to explore new possibilities.
*   **Inclusivity**: We aim to make machine learning more inclusive and accessible to individuals and organizations from diverse backgrounds.

**What We Offer**
-----------------

*   **2 Million+ Models**: Browse and explore a vast repository of pre-trained models covering various domains and use cases.
*   **Collaboration Platform**: Host and collaborate on unlimited public models, datasets, and applications.
*   **Inference Providers**: Leverage our inference providers to integrate our models into your applications with minimal latency and precision.
*   **Storage Buckets**: Securely store and manage your data with our scalable and reliable storage solutions.
*   **Enterprise Support**: Enjoy tailored support and resources specifically designed for enterprises and businesses.

**Join Our Community**
---------------------

*   **About Us**: Learn more about our mission, values, and history to understand the driving forces behind our platform.
*   **Blog**: Stay updated with the latest news, insights, and research on machine learning, AI, and related topics.
*   **Discord**: Connect with our vibrant community of users, developers, and researchers to share knowledge, ask questions, and get support.

**Careers**
---------

*   **Join Our Team**: We're looking for talented individuals to join our team and contribute to the growth and success of Hugging Face.

[Learn More](link to careers page)

By joining Hugging Face, you'll become part of a community that's shaping the future of machine learning and AI. Stay ahead of the curve and explore the endless possibilities of Hugging Face today.

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [55]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [82]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling llama3.2:3b
Found 10 relevant links


# Brochure of Hugging Face

## Who We Are

Hugging Face is the collaboration platform for the machine learning community. We empower the next generation of machine learning engineers, scientists, and end users to learn, collaborate, and share their work to build an open and ethical AI future together.

## Our Mission

Our mission is to make AI more accessible and usable for everyone. We provide a platform for the machine learning community to share, explore, discover, and experiment with open-source ML models, datasets, and applications.

## What We Do

* Host and collaborate on unlimited public models, datasets, and applications
* Provide a central place for machine learning engineers, scientists, and end users to learn, collaborate, and share their work
* Support the development of open-source ML libraries and tools
* Explore the edge of tech and drive innovation in the field of AI

## Our Community

Our community is growing rapidly and includes some of the most used open-source ML libraries and tools. Our talented science team is always exploring new ways to push the boundaries of AI.

## Our Solutions

* HuggingChat: a chat app powered by open-source AI models
* Hugging Face Hub: a central place for machine learning engineers, scientists, and end users to learn, collaborate, and share their work
* Inference Endpoints: a service that provides pre-trained models for inference
* Storage Buckets: a service that provides secure and reliable storage for models and datasets

## Our Values

* Collaboration: we believe that collaboration is key to accelerating the development of open-source ML
* Openness: we are committed to making AI more accessible and usable for everyone
* Innovation: we are always exploring new ways to push the boundaries of AI
* Ethics: we believe that AI should be developed and used in an ethical and responsible manner

## Get Involved

Join our community today and start contributing to the development of open-source ML. Whether you are a machine learning engineer, scientist, or end user, we want to hear from you and see what you can contribute.

## Contact Us

Learn more about Hugging Face and our mission at [website URL]. Contact us at [email address] or [LinkedIn profile URL] to get in touch with our team.

## Join the Conversation

Follow us on social media to stay up-to-date on the latest news and developments from Hugging Face.

* Twitter: [Twitter handle]
* LinkedIn: [LinkedIn profile URL]
* GitHub: [GitHub profile URL]
* Discord: [Discord server URL]

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>